# Auxiliary data sources

Collects the supporting Steam endpoints: store metadata, current player count, and the developer news feed.

# Setup

In [14]:
import requests
import json
import time
import random
import threading
import datetime as dt
from concurrent.futures import ThreadPoolExecutor,as_completed
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

project_root=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
catalogue_dir=project_root/'data'/'processed'/'catalogue'
meta_dir=project_root/'data'/'raw'/'meta'
ccu_dir=project_root/'data'/'raw'/'ccu'
news_dir=project_root/'data'/'raw'/'news'
for d in (meta_dir,ccu_dir,news_dir):
    d.mkdir(parents=True,exist_ok=True)

request_headers={'User-Agent':'EA-Health-Scoring-research/1.0 (academic study; mastudsaurabh33@gmail.com)'}
thread_local_storage=threading.local()

def get_session():
    if not hasattr(thread_local_storage,'session'):
        thread_local_storage.session=requests.Session()
        thread_local_storage.session.headers.update(request_headers)
    return thread_local_storage.session

# appdetails is rate limited hard, so go slowly
# a re-run only fetches what is missing, since existing files are skipped
auxiliary_workers=2
seconds_between_requests=1.2
news_items_per_game=100

def get_json(url,params,max_attempts=4):
    for attempt in range(max_attempts):
        try:
            response=get_session().get(url,params=params,timeout=30)
            if response.status_code==429 or response.status_code>=500:
                time.sleep((2**attempt)+random.random())
                continue
            parsed=response.json()
            time.sleep(seconds_between_requests)
            # a throttled response can be a valid 200 with a null body
            # catalogue notebook when a null body crashed the whole run at page 21
            return parsed if isinstance(parsed,dict) else None
        except (requests.RequestException,ValueError):
            time.sleep((2**attempt)+random.random())
    return None

candidates=pd.read_csv(catalogue_dir/'candidate_games.csv')
candidate_appids=candidates['appid'].tolist()
print('candidate games to enrich:',len(candidate_appids))

candidate games to enrich: 1887


# appdetails - genre, tags, price, developer, release date

In [15]:
def collect_appdetails(appid):
    output_file=meta_dir/f'{appid}.json'
    if output_file.exists():
        return 'skipped'

    payload=get_json('https://store.steampowered.com/api/appdetails',
                     {'appids':appid,'l':'english'})
    if not payload or not payload.get(str(appid),{}).get('success'):
        return 'failed'

    game=payload[str(appid)]['data']
    record={'appid':appid,
            'name':game.get('name'),
            'is_free':game.get('is_free'),
            'genres':[g['description'] for g in game.get('genres',[])],
            'categories':[c['description'] for c in game.get('categories',[])],
            'developers':game.get('developers'),
            'publishers':game.get('publishers'),
            'release_date_text':game.get('release_date',{}).get('date'),
            'coming_soon':game.get('release_date',{}).get('coming_soon'),
            'price_cents':game.get('price_overview',{}).get('initial'),
            'currency':game.get('price_overview',{}).get('currency'),
            'metacritic':game.get('metacritic',{}).get('score'),
            'required_age':game.get('required_age'),
            'collected_at':dt.datetime.now().isoformat(timespec='seconds')}
    output_file.write_text(json.dumps(record,indent=2,ensure_ascii=False),encoding='utf-8')
    return 'ok'


def run_pool(job,appids,label):
    counts={'ok':0,'skipped':0,'failed':0}
    started=time.time()
    with ThreadPoolExecutor(max_workers=auxiliary_workers) as pool:
        submitted={pool.submit(job,a):a for a in appids}
        for done_count,finished in enumerate(as_completed(submitted),1):
            try:
                counts[finished.result()]+=1
            except Exception as problem:
                counts['failed']+=1
                print(f'  app {submitted[finished]}: {type(problem).__name__}')
            if done_count%250==0:
                print(f'  {label} {done_count:,}/{len(appids):,}  ok={counts["ok"]:,} '
                      f'failed={counts["failed"]:,}',flush=True)
    print(f'{label} done in {(time.time()-started)/60:.1f} min: {counts}')
    return counts

print('Appdetails')
meta_counts=run_pool(collect_appdetails,candidate_appids,'appdetails')

Appdetails
  appdetails 250/1,887  ok=0 failed=0
  appdetails 500/1,887  ok=0 failed=0
  appdetails 750/1,887  ok=0 failed=0
  appdetails 1,000/1,887  ok=0 failed=0
  appdetails 1,250/1,887  ok=0 failed=0
  appdetails 1,500/1,887  ok=0 failed=0
  appdetails 1,750/1,887  ok=0 failed=0
appdetails done in 0.4 min: {'ok': 0, 'skipped': 1886, 'failed': 1}


# current player count

In [16]:
def collect_current_players(appid):
    output_file=ccu_dir/f'{appid}.json'
    if output_file.exists():
        return 'skipped'

    payload=get_json('https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/',
                     {'appid':appid})
    if not payload or payload.get('response',{}).get('result')!=1:
        return 'failed'

    record={'appid':appid,
            'current_players':payload['response'].get('player_count'),
            'measured_at':dt.datetime.now().isoformat(timespec='seconds')}
    output_file.write_text(json.dumps(record,indent=2),encoding='utf-8')
    return 'ok'

print('Current player count')
ccu_counts=run_pool(collect_current_players,candidate_appids,'ccu')

Current player count
  ccu 250/1,887  ok=0 failed=0
  ccu 500/1,887  ok=0 failed=0
  ccu 750/1,887  ok=0 failed=0
  ccu 1,000/1,887  ok=0 failed=0
  ccu 1,250/1,887  ok=0 failed=0
  ccu 1,500/1,887  ok=0 failed=0
  ccu 1,750/1,887  ok=0 failed=0
ccu done in 0.0 min: {'ok': 0, 'skipped': 1887, 'failed': 0}


# developer post cadence from the news feed

Developer posts are identified by the community feed, not by the external-url flag, which is true for press items too.

In [17]:
def collect_news(appid):
    output_file=news_dir/f'{appid}.json'
    if output_file.exists():
        return 'skipped'

    payload=get_json('https://api.steampowered.com/ISteamNews/GetNewsForApp/v2/',
                     {'appid':appid,'count':news_items_per_game,'maxlength':1})
    if not payload or 'appnews' not in payload:
        return 'failed'

    items=payload['appnews'].get('newsitems',[])

    # community feed, not is_external_url - that is true for press too
    # developer announcements, so it separates nothing
    developer_posts=[i for i in items if i.get('feedname')=='steam_community_announcements']
    press_posts=[i for i in items if i.get('feedname')!='steam_community_announcements']

    record={'appid':appid,
            'n_items_returned':len(items),
            'n_developer_posts':len(developer_posts),
            'n_press_posts':len(press_posts),
            'feedlabels_seen':sorted({i.get('feedlabel','') for i in items}),
            'developer_post_timestamps':[i.get('date') for i in developer_posts],
            'press_post_timestamps':[i.get('date') for i in press_posts],
            'collected_at':dt.datetime.now().isoformat(timespec='seconds')}
    output_file.write_text(json.dumps(record,indent=2,ensure_ascii=False),encoding='utf-8')
    return 'ok'

print('News feed / developer post cadence')
news_counts=run_pool(collect_news,candidate_appids,'news')

News feed / developer post cadence
  news 250/1,887  ok=0 failed=0
  news 500/1,887  ok=0 failed=0
  news 750/1,887  ok=0 failed=0
  news 1,000/1,887  ok=0 failed=0
  news 1,250/1,887  ok=0 failed=0
  news 1,500/1,887  ok=0 failed=0
  news 1,750/1,887  ok=0 failed=0
news done in 0.0 min: {'ok': 0, 'skipped': 1887, 'failed': 0}


In [18]:
# assertion cell. counts come off disk, not from the counters above
meta_files=list(meta_dir.glob('*.json'))
ccu_files=list(ccu_dir.glob('*.json'))
news_files=list(news_dir.glob('*.json'))

# i count coverage against the candidate list, so a file has to be for a game
# that is actually in the frame. counting every file on disk let leftovers from
# an earlier screening run push the share past 100%
wanted=set(int(a) for a in candidate_appids)


def coverage(files):
    have=set(int(f.stem) for f in files if f.stem.isdigit())
    return have&wanted, have-wanted


meta_have,meta_stale=coverage(meta_files)
ccu_have,ccu_stale=coverage(ccu_files)
news_have,news_stale=coverage(news_files)
denominator=max(len(wanted),1)

print('WHAT IS ACTUALLY ON DISK')
print(f'candidate games : {len(wanted):,}')
print(f'appdetails      : {len(meta_have):,}  ({100*len(meta_have)/denominator:.1f}%)')
print(f'current players : {len(ccu_have):,}  ({100*len(ccu_have)/denominator:.1f}%)')
print(f'news            : {len(news_have):,}  ({100*len(news_have)/denominator:.1f}%)')

# files for games the current screening did not keep. harmless, nothing reads
# them, but i would rather see them than have them inflate a percentage
stale=len(meta_stale)+len(ccu_stale)+len(news_stale)
if stale:
    print(f'\nleft over from an earlier run: {len(meta_stale):,} appdetails, '
          f'{len(ccu_stale):,} player counts, {len(news_stale):,} news')
    print('  not in the current candidate list, so nothing downstream reads them')

# how much of the feed is the developer rather than press
developer_post_counts=[]
press_ratio=[]
for f in news_files[:2000]:
    row=json.loads(f.read_text(encoding='utf-8'))
    developer_post_counts.append(row['n_developer_posts'])
    if row['n_items_returned']:
        press_ratio.append(row['n_press_posts']/row['n_items_returned'])

if developer_post_counts:
    s=pd.Series(developer_post_counts)
    print(f'\ndeveloper posts per game: median {s.median():.0f}, mean {s.mean():.1f}, max {s.max()}')
    print(f'games with zero developer posts: {(s==0).sum():,} of {len(s):,}')
    print(f'press share of the feed: median {pd.Series(press_ratio).median():.1%}')
    print('  ^ this is why filtering on is_external_url would have been wrong')

assert len(meta_files)>0, 'no appdetails collected at all'
assert len(ccu_files)>0, 'no player counts collected at all'
assert len(news_files)>0, 'no news collected at all'
print('\nALL CHECKS PASS')

WHAT IS ACTUALLY ON DISK
candidate games : 1,887
appdetails      : 1,886  (99.9%)
current players : 1,887  (100.0%)
news            : 1,887  (100.0%)

left over from an earlier run: 78 appdetails, 78 player counts, 78 news
  not in the current candidate list, so nothing downstream reads them

developer posts per game: median 65, mean 62.5, max 100
games with zero developer posts: 8 of 1,965
press share of the feed: median 0.0%
  ^ this is why filtering on is_external_url would have been wrong

ALL CHECKS PASS
